Load logic for `adwm_wh.gold.DimProduct`.

Business grain:

* One row per `ProductID`
* `ProductID` is the business key used by the merge
* Source duplicates are resolved before loading with a ranking step so only the best row per `ProductID` is used

Source tables:

* `adwm_wh.silver.product`
* `adwm_wh.silver.productsubcategory`
* `adwm_wh.silver.productcategory`

Notebook contents:

* Upsert logic for the gold dimension
* Validation summary comparing total rows to distinct `ProductID`
* Validation detail query that lists duplicate `ProductID` values only if any exist

Operational note:

* `modified_date` is refreshed only when tracked dimension attributes change

In [0]:
%sql
MERGE INTO adwm_wh.gold.DimProduct AS target
USING (
    WITH ranked_product_source AS (
        SELECT
            p.ProductID,
            p.ProductNumber,
            p.Name AS ProductName,
            NULLIF(p.Color, 'NA') AS Color,
            NULLIF(p.Size, 'NA') AS Size,
            p.StandardCost,
            p.ListPrice,
            p.ProductSubcategoryID,
            ROW_NUMBER() OVER (
                PARTITION BY p.ProductID
                ORDER BY
                    CASE WHEN p.ModifiedDate IS NOT NULL THEN 1 ELSE 0 END DESC,
                    CASE WHEN COALESCE(p.ProductSubcategoryID, 0) > 0 THEN 1 ELSE 0 END DESC,
                    CASE WHEN NULLIF(TRIM(p.ProductNumber), '') IS NOT NULL AND UPPER(TRIM(p.ProductNumber)) NOT IN ('NA', 'UNKNOWN') THEN 1 ELSE 0 END DESC,
                    CASE WHEN NULLIF(TRIM(p.Name), '') IS NOT NULL AND UPPER(TRIM(p.Name)) NOT IN ('NA', 'UNKNOWN') AND TRIM(p.Name) <> CAST(p.ProductID AS STRING) THEN 1 ELSE 0 END DESC,
                    CASE WHEN COALESCE(p.ListPrice, 0) > 0 THEN 1 ELSE 0 END DESC,
                    CASE WHEN COALESCE(p.StandardCost, 0) > 0 THEN 1 ELSE 0 END DESC,
                    p.ModifiedDate DESC,
                    p.ListPrice DESC,
                    p.StandardCost DESC,
                    p.ProductNumber DESC,
                    p.Name DESC
            ) AS rn
        FROM adwm_wh.silver.product p
    )
    SELECT
        p.ProductID,
        p.ProductNumber,
        p.ProductName,
        p.Color,
        p.Size,
        p.StandardCost,
        p.ListPrice,
        psc.Name AS SubcategoryName,
        pc.Name AS CategoryName,
        sha2(
            concat_ws(
                '||',
                coalesce(cast(p.ProductID AS STRING), ''),
                coalesce(p.ProductNumber, ''),
                coalesce(p.ProductName, ''),
                coalesce(p.Color, ''),
                coalesce(p.Size, ''),
                coalesce(cast(p.StandardCost AS STRING), ''),
                coalesce(cast(p.ListPrice AS STRING), ''),
                coalesce(psc.Name, ''),
                coalesce(pc.Name, '')
            ),
            256
        ) AS row_hash
    FROM ranked_product_source p
    LEFT JOIN adwm_wh.silver.productsubcategory psc
        ON psc.ProductSubcategoryID = p.ProductSubcategoryID
    LEFT JOIN adwm_wh.silver.productcategory pc
        ON pc.ProductCategoryID = psc.ProductCategoryID
    WHERE p.rn = 1
) AS source
ON target.ProductID = source.ProductID
WHEN MATCHED AND sha2(
    concat_ws(
        '||',
        coalesce(cast(target.ProductID AS STRING), ''),
        coalesce(target.ProductNumber, ''),
        coalesce(target.ProductName, ''),
        coalesce(target.Color, ''),
        coalesce(target.Size, ''),
        coalesce(cast(target.StandardCost AS STRING), ''),
        coalesce(cast(target.ListPrice AS STRING), ''),
        coalesce(target.SubcategoryName, ''),
        coalesce(target.CategoryName, '')
    ),
    256
) <> source.row_hash THEN UPDATE SET
    target.ProductNumber = source.ProductNumber,
    target.ProductName = source.ProductName,
    target.Color = source.Color,
    target.Size = source.Size,
    target.StandardCost = source.StandardCost,
    target.ListPrice = source.ListPrice,
    target.SubcategoryName = source.SubcategoryName,
    target.CategoryName = source.CategoryName,
    target.modified_date = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    ProductID,
    ProductNumber,
    ProductName,
    Color,
    Size,
    StandardCost,
    ListPrice,
    SubcategoryName,
    CategoryName,
    modified_date
)
VALUES (
    source.ProductID,
    source.ProductNumber,
    source.ProductName,
    source.Color,
    source.Size,
    source.StandardCost,
    source.ListPrice,
    source.SubcategoryName,
    source.CategoryName,
    current_timestamp()
);

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ProductID) AS distinct_product_ids,
    COUNT(*) - COUNT(DISTINCT ProductID) AS duplicate_row_count
FROM adwm_wh.gold.DimProduct;

In [0]:
%sql
SELECT
    ProductID,
    COUNT(*) AS row_count,
    MIN(modified_date) AS first_modified_date,
    MAX(modified_date) AS last_modified_date
FROM adwm_wh.gold.DimProduct
GROUP BY ProductID
HAVING COUNT(*) > 1
ORDER BY row_count DESC, ProductID;